In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load dokumen regulasi EU 261
loader = TextLoader('../data/documents/eu_261_2004', encoding='utf-8')
docs = loader.load()

# 2. Potong teks jadi chunk berukuran 500 karakter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(docs)

print(f"Jumlah dokumen ter-load: {len(docs)}")
print(f"Jumlah chunk berhasil dibuat: {len(chunks)}")


C:\Users\Hp user\AppData\Local\Temp\ipykernel_12592\3434581689.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Jumlah dokumen ter-load: 1
Jumlah chunk berhasil dibuat: 72


In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Inisialisasi model Embedding
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Konversi 72 chunk teks menjadi vektor & simpan di FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

# 3. indeks FAISS simpan ke lokal biar bisa dipanggil kapan aja
vectorstore.save_local("../models/faiss_index")

print("Vector Store FAISS berhasil dibuat & disimpan di folder /models/faiss_index!")


C:\Users\Hp user\AppData\Local\Temp\ipykernel_12592\2207092137.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\Hp user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 386.10it/s]


Vector Store FAISS berhasil dibuat & disimpan di folder /models/faiss_index!


In [3]:
# Tes query pencarian regulasi
query = "How much compensation for delayed flight?"
docs = vectorstore.similarity_search(query, k=2)

print("--- Hasil Pencarian Regulasi ---")
for i, doc in enumerate(docs):
    print(f"\n[Hasil {i+1}]:")
    print(doc.page_content)


--- Hasil Pencarian Regulasi ---

[Hasil 1]:
2. In any event, the assistance shall be offered within the time limits set out above with respect to each distance bracket.

Article 7

Right to compensation

1. Where reference is made to this Article, passengers shall receive compensation amounting to:

(a) EUR 250 for all flights of 1500 kilometres or less;

(b) EUR 400 for all intra-Community flights of more than 1500 kilometres, and for all other flights between 1500 and 3500 kilometres;

[Hasil 2]:
(b) be offered assistance by the operating air carrier in accordance with Article 9(1)(a) and 9(2), as well as, in event of re-routing when the reasonably expected time of departure of the new flight is at least the day after the departure as it was planned for the cancelled flight, the assistance specified in Article 9(1)(b) and 9(1)(c); and

(c) have the right to compensation by the operating air carrier in accordance with Article 7, unless:


In [4]:
import joblib
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#load Model ML dan Vector Store
model = joblib.load('../models/lgb_model.joblib')
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local("../models/faiss_index", embeddings, allow_dangerous_deserialization=True)

# 1. 7 data fitur penerbangan
sample_df = pd.DataFrame([{
    'MONTH': 1,
    'DAY_OF_WEEK': 3,
    'AIRLINE': 5,
    'ORIGIN_AIRPORT': 102,
    'DESTINATION_AIRPORT': 205,
    'dep_hour': 14,
    'DISTANCE': 1200
}])

# 2. Prediksi Resiko Delay
delay_prob = model.predict_proba(sample_df.values)[0][1]

# 3. Cari Regulasi EU 261 dengan FAISS RAG
user_question = "What compensation am I entitled to?"
relevant_docs = vectorstore.similarity_search(user_question, k=1)

# 4. Tampilkan Laporan Hasil
print("FLIGHT DELAY ADVISOR REPORT")
print(f"Probabilitas Delay: {delay_prob * 100:.1f}%")
print(f"Status Risiko     : {'RISIKO TINGGI DELAY' if delay_prob > 0.5 else 'RISIKO RENDAH (ON-TIME)'}")
print("\n--- Regulasi Hak Penumpang (EU 261) ---")
print(relevant_docs[0].page_content)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 337.09it/s]


FLIGHT DELAY ADVISOR REPORT
Probabilitas Delay: 24.1%
Status Risiko     : RISIKO RENDAH (ON-TIME)

--- Regulasi Hak Penumpang (EU 261) ---
2. In any event, the assistance shall be offered within the time limits set out above with respect to each distance bracket.

Article 7

Right to compensation

1. Where reference is made to this Article, passengers shall receive compensation amounting to:

(a) EUR 250 for all flights of 1500 kilometres or less;

(b) EUR 400 for all intra-Community flights of more than 1500 kilometres, and for all other flights between 1500 and 3500 kilometres;


c:\Users\Hp user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
